# HW14: Эмбеддинги, FAISS, оценка retrieval и mini-RAG

**Задача**: Построить полную mini-RAG систему с векторным поиском (FAISS), оценкой качества retrieval, обновлением базы знаний и анализом ошибок.

In [10]:
# ============================================================================
# Секция 2.3.1: Импорты, seed и конфигурация среды
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import json
import os
from typing import List, Dict, Tuple
from pathlib import Path
from sklearn import metrics

# Установка seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Импорт faiss
try:
    import faiss
    print(f"✓ faiss успешно импортирован")
except ImportError:
    print("Установка faiss-cpu...")
    os.system("pip install faiss-cpu -q")
    import faiss

# Импорт sentence transformers для эмбеддингов
try:
    from sentence_transformers import SentenceTransformer
    print(f"✓ sentence_transformers успешно импортирован")
except ImportError:
    print("Установка sentence-transformers...")
    os.system("pip install sentence-transformers -q")
    from sentence_transformers import SentenceTransformer

# Определение устройства
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    print(f"✓ torch доступен, используется устройство: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print(f"Используется устройство: {DEVICE}")

# Создание директории для артефактов
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"✓ Директория артефактов: {ARTIFACTS_DIR.absolute()}")

print("\n=== КОНФИГУРАЦИЯ СРЕДЫ ЗАВЕРШЕНА ===")

✓ faiss успешно импортирован
✓ sentence_transformers успешно импортирован
✓ torch доступен, используется устройство: cpu
✓ Директория артефактов: c:\Users\fahitalis\Courses\mirea-aie\mirea-aie-student\homeworks\HW14\artifacts

=== КОНФИГУРАЦИЯ СРЕДЫ ЗАВЕРШЕНА ===


## Секция 2.3.2: База знаний - FAQ по эмбеддингам и векторному поиску

**Предметная область**: Machine Learning / NLP - Эмбеддинги, семантический поиск и индексирование FAISS

**Почему эта область**: Тематически связана с курсом и предоставляет хороший материал для задач retrieval. Вопросы об эмбеддингах, индексировании и семантическом поиске хорошо подходят для векторного retrieval.

In [11]:
# ============================================================================
# Загрузка базы знаний: FAQ об эмбеддингах и векторном поиске
# ============================================================================

# Создание базы знаний из документов
documents = [
    {
        "id": 1,
        "title": "Что такое word embeddings и почему они важны?",
        "content": "Word embeddings (словесные эмбеддинги) - это плотные векторные представления слов в непрерывном векторном пространстве. Каждое слово представляется вектором из вещественных чисел. Эмбеддинги захватывают семантическое значение: слова с похожим значением имеют векторы, которые близко расположены в векторном пространстве. Это свойство делает эмбеддинги полезными для задач обработки естественного языка. Популярные методы создания эмбеддингов включают Word2Vec, GloVe и FastText. Нейронные сети могут автоматически обучать эмбеддинги через тренировку на языковых задачах."
    },
    {
        "id": 2,
        "title": "Чем отличаются sentence embeddings от word embeddings?",
        "content": "Sentence embeddings (эмбеддинги предложений) представляют целые предложения одним вектором, захватывая семантическое значение всего предложения. Хотя word embeddings представляют отдельные слова, sentence embeddings можно создавать путем усреднения word embeddings или использования более сложных методов, таких как модели Transformer. Sentence embeddings полезны для сравнения сходства между предложениями, кластеризации документов и семантического поиска. Современные sentence embeddings из моделей, таких как BERT и Sentence-BERT, производят лучшие представления, потому что они понимают контекст и грамматику, выходящие за пределы просто значения слова."
    },
    {
        "id": 3,
        "title": "Что такое FAISS и почему он используется в поиске сходства?",
        "content": "FAISS (Facebook AI Similarity Search) - это библиотека, которая предоставляет эффективный поиск сходства и кластеризацию плотных векторов. Она разработана для обработки миллиардов векторов, даже когда они не помещаются в ОЗУ. FAISS работает, построив структуру индекса, которая организует векторы для быстрого извлечения. Вместо вычисления сходства с каждым вектором (сложность O(n)), FAISS использует приблизительный поиск ближайших соседей, достигая суб-линейной сложности. Типичные типы индексов FAISS включают плоские индексы, квантизацию произведений и графы навигируемые малые миры. FAISS широко используется в системах рекомендаций, семантическом поиске и retrieval-augmented generation."
    },
    {
        "id": 4,
        "title": "Что такое cosine similarity и как она используется в retrieval?",
        "content": "Cosine similarity (косинусное сходство) измеряет угол между двумя векторами. Она варьируется от -1 до 1, где 1 означает идентичное направление (наиболее похоже) и -1 означает противоположное направление. Для эмбеддингов мы обычно используем векторы в положительном пространстве, поэтому косинусное сходство варьируется от 0 до 1. Косинусное сходство инвариантно к величине вектора, что означает, что важно только направление, но не длина. Вот почему она предпочтительнее евклидова расстояния для высокомерных эмбеддингов. В задачах retrieval мы находим документы, чьи эмбеддинги имеют наивысшее косинусное сходство с эмбеддингом запроса. Это дает нам наиболее семантически похожие документы."
    },
    {
        "id": 5,
        "title": "Как работает семантический поиск?",
        "content": "Семантический поиск находит документы на основе значения, а не на совпадении ключевых слов. Процесс включает следующие шаги: (1) преобразование запроса пользователя в эмбеддинг с помощью модели эмбеддинга, (2) вычисление сходства между эмбеддингом запроса и эмбеддингами документов, (3) ранжирование документов по баллу сходства, (4) возврат топ-k наиболее похожих документов. Ключевное преимущество в том, что синонимы и семантически связанные термины найдены, даже без точного совпадения ключевых слов. Например, запрос 'глубокое обучение модели' найдет документы о 'нейронных сетях' или 'трансформерах', даже если эти точные слова не появляются в запросе."
    },
    {
        "id": 6,
        "title": "في разница между плотными и разреженными эмбеддингами?",
        "content": "Плотные эмбеддинги (dense embeddings) - это векторы, где большинство или все значения ненулевые. Они компактны, эффективны и захватывают непрерывную семантическую информацию. Примеры включают эмбеддинги Word2Vec, GloVe и BERT. Разреженные эмбеддинги (sparse embeddings) имеют в основном нули с несколькими ненулевыми значениями. Они часто соответствуют индексам активных признаков, таких как в bag-of-words представлениях или TF-IDF векторах. Плотные эмбеддинги в целом предпочтительны для нейрального retrieval, потому что они более компактны и лучше работают с векторными базами данных. Однако разреженные эмбеддинги могут быть более интерпретируемы и хорошо работают с традиционными методами ранжирования. Гибридные подходы объединяют оба плотных и разреженных представления."
    },
    {
        "id": 7,
        "title": "Что такое Retrieval-Augmented Generation (RAG) и как это работает?",
        "content": "Retrieval-Augmented Generation (RAG) - это техника, которая объединяет поиск информации с генерацией текста. Конвейер RAG имеет следующие шаги: (1) извлечение релевантных документов или отрывков из базы знаний с использованием семантического сходства, (2) конкатенация извлеченных отрывков как контекста, (3) передача запроса и контекста в языковую модель, (4) LLM генерирует ответ, основанный на извлеченном контексте. RAG улучшает сгенерированные ответы, предоставляя фактический контекст из надежных источников. Это уменьшает галлюцинации по сравнению с LLM, отвечающими без контекста. RAG полезен для ответов на вопросы по документам, ответов на вопросы на основе фактов и привязки выходов LLM к конкретным знаниям."
    },
    {
        "id": 8,
        "title": "Как вы оцениваете качество retrieval?",
        "content": "Качество retrieval обычно оценивается с использованием метрик, таких как: (1) Precision@k - доля топ-k извлеченных элементов, которые релевантны, (2) Recall@k - доля всех релевантных элементов, которые появляются в топ-k результатах, (3) Mean Reciprocal Rank (MRR) - среднее обратное ранжирования первого релевантного результата, (4) NDCG (Normalized Discounted Cumulative Gain) - учитывает позицию ранжирования и релевантность. Для оценки вам нужен размеченный набор тестовых запросов с наземной истиной релевантных документов. Вы запускаете retrieval на этих запросах и измеряете, сколько релевантных документов появляется в топ-k результатах. Для хорошей системы семантического поиска вам требуется высокий recall и precision при малых значениях k (например, топ 5 или топ 10)."
    },
    {
        "id": 9,
        "title": "Что такое документные чанки и почему мы их используем?",
        "content": "Документные чанки (chunks) или отрывки - это сегменты более длинных документов. Вместо создания эмбеддингов для целых документов, мы часто разбиваем их на меньшие чанки по нескольким причинам: (1) лучшая зернистость retrieval - мы извлекаем конкретные релевантные отрывки, а не целые документы, (2) модели эмбеддинга имеют ограничения на длину входных данных (обычно 512 токена), (3) меньшие чанки более вероятны, чтобы соответствовать намерению запроса, (4) эффективность - меньше векторов для индексирования и поиска. Стратегии чанкирования включают: чанки фиксированного размера, перекрывающиеся чанки (для сохранения контекста), семантическое чанкирование на основе предложений или абзацев и иерархическое чанкирование. Размер чанка и перекрытие - это важные гиперпараметры, влияющие на качество retrieval."
    },
    {
        "id": 10,
        "title": "В чем разница между приблизительным и точным поиском ближайшего соседа?",
        "content": "Точный поиск ближайшего соседа (exact nearest neighbor search) гарантирует нахождение истинных k-ближайших соседей путем сравнения вектора запроса со всеми векторами в базе данных. Это точно, но медленно для больших баз данных (сложность O(n)). Приблизительный поиск ближайшего соседа (approximate nearest neighbor search) использует умные структуры данных для поиска очень похожих соседей без проверки всех векторов, достигая суб-линейной сложности. Компромисс: приблизительные методы намного быстрее, но могут пропустить истинных k-ближайших соседей. Ошибка приближения зависит от параметров индекса. Для большинства практических приложений предпочтительны приблизительные методы, потому что скорость более важна, чем нахождение точных k-ближайших соседей. Методы включают: квантизацию произведений, локально-чувствительное хеширование и навигируемые графы малые миры."
    },
    {
        "id": 11,
        "title": "Как работает модель Sentence-BERT?",
        "content": "Sentence-BERT (SBERT) - это модификация BERT, разработанная для производства хороших sentence embeddings. Хотя BERT использует сложные стратегии объединения, которые не производят отличные представления на уровне предложий, SBERT тренирует BERT с использованием сиамских и триплетных структур сетей для прямого производства хороших sentence embeddings. Модель использует операцию объединения (обычно среднее) над эмбеддингами токенов для создания вектора предложения фиксированного размера. SBERT обучается на управляемых наборах данных (NLI, STS) для изучения значимых эмбеддингов, где сходство в векторном пространстве коррелирует с семантическим сходством. Предварительно обученные модели SBERT доступны для многих языков и доменов. Они эффективны и производят лучшие эмбеддинги, чем усреднение эмбеддингов токенов BERT."
    },
    {
        "id": 12,
        "title": "Какие факторы влияют на качество эмбеддингов и производительность retrieval?",
        "content": "Несколько факторов влияют на качество эмбеддингов и retrieval: (1) выбор модели эмбеддинга - разные модели захватывают разные семантические аспекты, (2) домен данных обучения - модели, обученные на похожих доменах, работают лучше, (3) предварительная обработка входного текста - токенизация и очистка влияют на качество эмбеддинга, (4) стратегия чанкирования документов - размер чанка и выбор границ важны, (5) метрика сходства - косинусное сходство vs евклидово расстояние vs другие метрики, (6) тип индекса - разные индексы FAISS имеют разные компромиссы точность/скорость, (7) гиперпараметры retrieval - значение top-k влияет на компромисс precision-recall. Для лучших результатов выбирайте модель эмбеддинга, обученную на похожем тексте, используйте подходящее чанкирование и настраивайте параметры retrieval на основе метрик оценки."
    }
]

print(f"✓ База знаний создана: {len(documents)} документов")
print("\nПримеры документов:")
for doc in documents[:3]:
    print(f"  [{doc['id']}] {doc['title']}")
print(f"  ... и еще {len(documents)-3} документов\n")

# Вывод статистики
total_chars = sum(len(doc['content']) for doc in documents)
avg_chars = total_chars / len(documents)
print(f"Статистика базы знаний:")
print(f"  Всего документов: {len(documents)}")
print(f"  Всего символов: {total_chars}")
print(f"  Среднее число символов на документ: {avg_chars:.0f}")

✓ База знаний создана: 12 документов

Примеры документов:
  [1] Что такое word embeddings и почему они важны?
  [2] Чем отличаются sentence embeddings от word embeddings?
  [3] Что такое FAISS и почему он используется в поиске сходства?
  ... и еще 9 документов

Статистика базы знаний:
  Всего документов: 12
  Всего символов: 8887
  Среднее число символов на документ: 741


## Секция 2.3.3: Разбиение документов на чанки

Стратегия: Разбить документы на перекрывающиеся чанки для сохранения контекста при сохранении индивидуальных чанков, управляемых для модели эмбеддинга.

In [12]:
# ============================================================================
# Реализация чанкирования
# ============================================================================

CHUNK_SIZE = 256  # символов на чанк
CHUNK_OVERLAP = 50  # перекрытие между чанками
MIN_CHUNK_SIZE = 50  # минимальный размер чанка

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """
    Разбить текст на перекрывающиеся чанки.
    
    Args:
        text: Входной текст для разбиения
        chunk_size: Целевой размер каждого чанка в символах
        overlap: Число символов для перекрытия между чанками
    
    Returns:
        Список текстовых чанков
    """
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        
        # Убедиться в минимальном размере чанка
        if len(chunk) >= MIN_CHUNK_SIZE or start == 0:
            chunks.append(chunk)
        
        # Переместить начальную позицию для следующего чанка
        start = end - overlap
        
        # Остановиться, если мы достигли конца
        if end == len(text):
            break
    
    return chunks

# Создать чанки для всех документов
chunks_list = []  # Список (doc_id, title, chunk_text)
chunk_metadata = []  # Для справки

for doc in documents:
    chunks = chunk_text(doc['content'], CHUNK_SIZE, CHUNK_OVERLAP)
    for chunk_idx, chunk in enumerate(chunks):
        chunks_list.append({
            'doc_id': doc['id'],
            'title': doc['title'],
            'chunk_idx': chunk_idx,
            'text': chunk,
            'source': f"Doc{doc['id']}_Chunk{chunk_idx}"
        })

print(f"✓ Разбиение на чанки завершено")
print(f"  Всего созданно чанков: {len(chunks_list)}")
print(f"  Размер чанка: {CHUNK_SIZE} символов")
print(f"  Перекрытие: {CHUNK_OVERLAP} символов")

# Показать пример: как один документ становится чанками
example_doc_id = 1
example_chunks = [c for c in chunks_list if c['doc_id'] == example_doc_id]
print(f"\nПример: Документ {example_doc_id} '{documents[example_doc_id-1]['title']}'")
print(f"  Исходная длина: {len(documents[example_doc_id-1]['content'])} символов")
print(f"  Число чанков: {len(example_chunks)}")
print(f"  Отрывок из первого чанка: {example_chunks[0]['text'][:100]}...")
if len(example_chunks) > 1:
    print(f"  Отрывок из второго чанка: {example_chunks[1]['text'][:100]}...")

✓ Разбиение на чанки завершено
  Всего созданно чанков: 45
  Размер чанка: 256 символов
  Перекрытие: 50 символов

Пример: Документ 1 'Что такое word embeddings и почему они важны?'
  Исходная длина: 570 символов
  Число чанков: 3
  Отрывок из первого чанка: Word embeddings (словесные эмбеддинги) - это плотные векторные представления слов в непрерывном вект...
  Отрывок из второго чанка: антическое значение: слова с похожим значением имеют векторы, которые близко расположены в векторном...


## Секция 2.3.4: Эмбеддинги и индекс FAISS

Загрузить предварительно обученную модель Sentence-BERT и построить индекс FAISS для эффективного семантического поиска.

In [13]:
# ============================================================================
# Генерация эмбеддингов
# ============================================================================

print("Загрузка модели Sentence-BERT...")
# Используем меньшую, более быструю модель, подходящую для CPU
model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f"✓ Модель загружена на устройство: {DEVICE}")
print(f"  Размер эмбеддинга модели: {model.get_sentence_embedding_dimension()}")

# Генерация эмбеддингов для всех чанков
print("\nГенерация эмбеддингов для всех чанков...")
chunk_texts = [chunk['text'] for chunk in chunks_list]
embeddings = model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"✓ Эмбеддинги сгенерированы")
print(f"  Форма: {embeddings.shape}")
print(f"  Тип данных: {embeddings.dtype}")

# ============================================================================
# Построение индекса FAISS
# ============================================================================

print("\nПостроение индекса FAISS...")
embedding_dim = embeddings.shape[1]

# Используем L2 (Евклидово) расстояние индекс - может быть преобразовано в сходство
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)

print(f"✓ Индекс FAISS построен")
print(f"  Тип индекса: IndexFlatL2")
print(f"  Размерность: {embedding_dim}")
print(f"  Число векторов: {index.ntotal}")
print(f"  Метрика: L2 расстояние (меньше = лучше/более похоже)")

# ============================================================================
# Пример retrieval
# ============================================================================

TOP_K = 5

def retrieve(query: str, k: int = TOP_K) -> List[Dict]:
    """
    Извлечь топ-k чанков, наиболее похожих на запрос.
    
    Returns:
        Список словарей с информацией о чанке и баллом сходства
    """
    # Кодировать запрос
    query_embedding = model.encode([query], convert_to_numpy=True)
    
    # Поиск в индексе FAISS
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for idx, (distance, retrieved_idx) in enumerate(zip(distances[0], indices[0])):
        if retrieved_idx >= 0:  # -1 означает невалидный
            chunk = chunks_list[retrieved_idx]
            # Преобразовать L2 расстояние в сходство (не точное косинусное, но понятное)
            similarity = 1.0 / (1.0 + distance)  # Простое преобразование
            results.append({
                'rank': idx + 1,
                'chunk_idx': retrieved_idx,
                'source': chunk['source'],
                'doc_id': chunk['doc_id'],
                'title': chunk['title'],
                'text': chunk['text'],
                'distance': float(distance),
                'similarity': float(similarity)
            })
    
    return results

# Тестирование retrieval
print("\n" + "="*70)
print("ПРИМЕРЫ ТЕСТИРОВАНИЯ RETRIEVAL")
print("="*70)

test_queries = [
    "Что такое эмбеддинги?",
    "Как работает FAISS?",
    "Расскажи о семантическом поиске"
]

for query in test_queries:
    print(f"\nЗапрос: '{query}'")
    results = retrieve(query, k=3)
    for res in results:
        print(f"  [{res['rank']}] {res['source']} (сход: {res['similarity']:.3f})")
        print(f"      Заголовок: {res['title'][:60]}...")
        print(f"      Текст: {res['text'][:80]}...")

Загрузка модели Sentence-BERT...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Модель загружена на устройство: cpu
  Размер эмбеддинга модели: 384

Генерация эмбеддингов для всех чанков...


C:\Users\fahitalis\AppData\Local\Temp\ipykernel_17412\1926094834.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Размер эмбеддинга модели: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Эмбеддинги сгенерированы
  Форма: (45, 384)
  Тип данных: float32

Построение индекса FAISS...
✓ Индекс FAISS построен
  Тип индекса: IndexFlatL2
  Размерность: 384
  Число векторов: 45
  Метрика: L2 расстояние (меньше = лучше/более похоже)

ПРИМЕРЫ ТЕСТИРОВАНИЯ RETRIEVAL

Запрос: 'Что такое эмбеддинги?'
  [1] Doc4_Chunk3 (сход: 0.610)
      Заголовок: Что такое cosine similarity и как она используется в retriev...
      Текст: мбеддингом запроса. Это дает нам наиболее семантически похожие документы....
  [2] Doc1_Chunk2 (сход: 0.498)
      Заголовок: Что такое word embeddings и почему они важны?...
      Текст: е методы создания эмбеддингов включают Word2Vec, GloVe и FastText. Нейронные сет...
  [3] Doc4_Chunk2 (сход: 0.493)
      Заголовок: Что такое cosine similarity и как она используется в retriev...
      Текст:  направление, но не длина. Вот почему она предпочтительнее евклидова расстояния ...

Запрос: 'Как работает FAISS?'
  [1] Doc4_Chunk3 (сход: 0.537)
      Заголовок: Что 

## Секция 2.3.5: Контрольные запросы и оценка качества retrieval

Подготовить контрольные запросы с ожидаемыми релевантными источниками и оценить, используя метрики hit@k и recall@k.

In [14]:
# ============================================================================
# Определение контрольных запросов с ожидаемыми релевантными документами
# ============================================================================

control_queries = [
    {
        "query": "Для чего используются word embeddings?",
        "expected_doc_ids": [1],  # Документ 1 о word embeddings
        "description": "Прямой вопрос об эмбеддингах"
    },
    {
        "query": "Как выполнить семантический поиск с векторами?",
        "expected_doc_ids": [3, 4, 5],  # FAISS, cosine similarity, semantic search
        "description": "О векторном сходстве и retrieval"
    },
    {
        "query": "Расскажи о FAISS",
        "expected_doc_ids": [3],  # Документ 3 специально о FAISS
        "description": "Прямой вопрос о FAISS"
    },
    {
        "query": "Как я могу оценить, хорошо ли работает моя система retrieval?",
        "expected_doc_ids": [8],  # Документ 8 об оценке retrieval
        "description": "О метриках оценки"
    },
    {
        "query": "Что такое RAG и как он объединяет retrieval с генерацией?",
        "expected_doc_ids": [7],  # Документ 7 о RAG
        "description": "О retrieval-augmented generation"
    },
    {
        "query": "Объясни разницу между плотными и разреженными эмбеддингами",
        "expected_doc_ids": [6],  # Документ 6 о плотных vs разреженных
        "description": "Вопрос на сравнение"
    },
    {
        "query": "Почему нам нужно разбивать документы на чанки?",
        "expected_doc_ids": [9],  # Документ 9 о чанкировании
        "description": "О разбиении документов на чанки"
    },
    {
        "query": "В чем разница между точным и приблизительным поиском ближайшего соседа?",
        "expected_doc_ids": [10],  # Документ 10 об этом
        "description": "О точном vs приблизительном поиске"
    },
    {
        "query": "Как модель Sentence-BERT производит хорошие sentence embeddings?",
        "expected_doc_ids": [11],  # Документ 11 о Sentence-BERT
        "description": "Об архитектуре SBERT"
    },
    {
        "query": "Какие факторы влияют на качество эмбеддингов?",
        "expected_doc_ids": [12],  # Документ 12 о факторах
        "description": "О факторах качества"
    },
]

print(f"✓ Создано {len(control_queries)} контрольных запросов")

# ============================================================================
# Метрики оценки retrieval
# ============================================================================

def evaluate_retrieval(query: str, expected_doc_ids: List[int], retrieved_chunks: List[Dict], k: int = TOP_K):
    """
    Оценить качество retrieval для одного запроса.
    
    Метрики:
    - hit@k: найден ли хотя бы один релевантный документ в топ-k
    - recall@k: доля найденных релевантных документов из всех ожидаемых
    - mrr@k: mean reciprocal rank первого релевантного документа
    """
    retrieved_doc_ids = [chunk['doc_id'] for chunk in retrieved_chunks]
    
    # Hit@k: Найден ли хотя бы один релевантный документ?
    hit_at_k = any(doc_id in retrieved_doc_ids for doc_id in expected_doc_ids)
    
    # Recall@k: Какую долю релевантных документов мы нашли?
    found = sum(1 for doc_id in expected_doc_ids if doc_id in retrieved_doc_ids)
    recall_at_k = found / len(expected_doc_ids) if expected_doc_ids else 0
    
    # MRR@k: Позиция первого релевантного документа
    mrr_at_k = 0
    for rank, chunk in enumerate(retrieved_chunks, 1):
        if chunk['doc_id'] in expected_doc_ids:
            mrr_at_k = 1.0 / rank
            break
    
    return {
        'hit_at_k': hit_at_k,
        'recall_at_k': recall_at_k,
        'mrr_at_k': mrr_at_k,
        'found_count': found,
        'expected_count': len(expected_doc_ids)
    }

# ============================================================================
# Запуск оценки на контрольных запросах
# ============================================================================

print("\n" + "="*70)
print("ОЦЕНКА RETRIEVAL НА КОНТРОЛЬНЫХ ЗАПРОСАХ")
print("="*70)

evaluation_results = []

for query_info in control_queries:
    query = query_info['query']
    expected_doc_ids = query_info['expected_doc_ids']
    
    # Извлечь
    retrieved = retrieve(query, k=TOP_K)
    
    # Оценить
    metrics = evaluate_retrieval(query, expected_doc_ids, retrieved, TOP_K)
    
    # Сохранить результаты
    result_record = {
        'query': query,
        'expected_source': ', '.join(f"Doc{doc_id}" for doc_id in expected_doc_ids),
        'retrieved_sources': ', '.join(r['source'] for r in retrieved),
        'hit_at_k': metrics['hit_at_k'],
        'recall_at_k': metrics['recall_at_k'],
        'mrr_at_k': metrics['mrr_at_k'],
        'rank_of_first_relevant': None
    }
    
    # Найти ранжирование первого релевантного
    for rank, chunk in enumerate(retrieved, 1):
        if chunk['doc_id'] in expected_doc_ids:
            result_record['rank_of_first_relevant'] = rank
            break
    
    evaluation_results.append(result_record)
    
    print(f"\nЗапрос: {query}")
    print(f"  Ожидаемые документы: {result_record['expected_source']}")
    print(f"  Hit@{TOP_K}: {metrics['hit_at_k']}, Recall@{TOP_K}: {metrics['recall_at_k']:.2f}, MRR: {metrics['mrr_at_k']:.2f}")

# Создать dataframe оценки
eval_df = pd.DataFrame(evaluation_results)

# Вычислить общие метрики
hit_rate = eval_df['hit_at_k'].mean()
recall_rate = eval_df['recall_at_k'].mean()
mrr_rate = eval_df['mrr_at_k'].mean()

print(f"\n" + "="*70)
print(f"ОБЩИЕ МЕТРИКИ RETRIEVAL")
print(f"="*70)
print(f"Hit@{TOP_K}: {hit_rate:.2%} ({eval_df['hit_at_k'].sum()}/{len(eval_df)})")
print(f"Recall@{TOP_K}: {recall_rate:.2%}")
print(f"MRR@{TOP_K}: {mrr_rate:.3f}")

# Сохранить результаты оценки
eval_df.to_csv(ARTIFACTS_DIR / "retrieval_eval.csv", index=False)
print(f"\n✓ Оценка сохранена в artifacts/retrieval_eval.csv")

✓ Создано 10 контрольных запросов

ОЦЕНКА RETRIEVAL НА КОНТРОЛЬНЫХ ЗАПРОСАХ

Запрос: Для чего используются word embeddings?
  Ожидаемые документы: Doc1
  Hit@5: True, Recall@5: 1.00, MRR: 0.20

Запрос: Как выполнить семантический поиск с векторами?
  Ожидаемые документы: Doc3, Doc4, Doc5
  Hit@5: True, Recall@5: 0.33, MRR: 1.00

Запрос: Расскажи о FAISS
  Ожидаемые документы: Doc3
  Hit@5: True, Recall@5: 1.00, MRR: 0.33

Запрос: Как я могу оценить, хорошо ли работает моя система retrieval?
  Ожидаемые документы: Doc8
  Hit@5: False, Recall@5: 0.00, MRR: 0.00

Запрос: Что такое RAG и как он объединяет retrieval с генерацией?
  Ожидаемые документы: Doc7
  Hit@5: True, Recall@5: 1.00, MRR: 0.50

Запрос: Объясни разницу между плотными и разреженными эмбеддингами
  Ожидаемые документы: Doc6
  Hit@5: True, Recall@5: 1.00, MRR: 1.00

Запрос: Почему нам нужно разбивать документы на чанки?
  Ожидаемые документы: Doc9
  Hit@5: True, Recall@5: 1.00, MRR: 0.50

Запрос: В чем разница между точным 

## Секция 2.3.6: Экспериментальное сравнение - Влияние размера чанка

Сравнить два размера чанков, чтобы посмотреть, как этот параметр влияет на качество retrieval.

In [15]:
# ============================================================================
# Эксперимент: Влияние размера чанка на качество retrieval
# ============================================================================

print("="*70)
print("ЭКСПЕРИМЕНТ: ВЛИЯНИЕ РАЗМЕРА ЧАНКА")
print("="*70)

def evaluate_chunk_size(chunk_size: int, overlap: int, query_subset=None):
    """Оценить retrieval с определенным размером чанка."""
    # Создать чанки с новым размером
    chunks_exp = []
    for doc in documents:
        chunks = chunk_text(doc['content'], chunk_size, overlap)
        for chunk_idx, chunk in enumerate(chunks):
            chunks_exp.append({
                'doc_id': doc['id'],
                'title': doc['title'],
                'chunk_idx': chunk_idx,
                'text': chunk,
                'source': f"Doc{doc['id']}_Chunk{chunk_idx}"
            })
    
    # Генерировать эмбеддинги
    chunk_texts_exp = [chunk['text'] for chunk in chunks_exp]
    embeddings_exp = model.encode(chunk_texts_exp, convert_to_numpy=True, show_progress_bar=False)
    
    # Построить индекс
    index_exp = faiss.IndexFlatL2(embeddings_exp.shape[1])
    index_exp.add(embeddings_exp)
    
    # Оценить
    def retrieve_exp(query):
        query_emb = model.encode([query], convert_to_numpy=True)
        distances, indices = index_exp.search(query_emb, TOP_K)
        results = []
        for distance, idx in zip(distances[0], indices[0]):
            if idx >= 0:
                chunk = chunks_exp[idx]
                results.append({
                    'doc_id': chunk['doc_id'],
                    'source': chunk['source']
                })
        return results
    
    # Использовать подмножество запросов, если указано
    queries_to_eval = query_subset if query_subset else control_queries
    
    hits = 0
    recalls = []
    for query_info in queries_to_eval:
        query = query_info['query']
        expected_doc_ids = query_info['expected_doc_ids']
        retrieved = retrieve_exp(query)
        retrieved_doc_ids = [r['doc_id'] for r in retrieved]
        
        # Hit
        hit = any(doc_id in retrieved_doc_ids for doc_id in expected_doc_ids)
        hits += hit
        
        # Recall
        found = sum(1 for doc_id in expected_doc_ids if doc_id in retrieved_doc_ids)
        recall = found / len(expected_doc_ids) if expected_doc_ids else 0
        recalls.append(recall)
    
    hit_rate = hits / len(queries_to_eval)
    recall_rate = np.mean(recalls)
    
    return {
        'chunk_size': chunk_size,
        'overlap': overlap,
        'num_chunks': len(chunks_exp),
        'hit_rate': hit_rate,
        'recall_rate': recall_rate
    }

# Сравнить два размера чанков
chunk_size_1 = 256  # Текущий
chunk_size_2 = 512  # Более крупный

results_1 = evaluate_chunk_size(chunk_size_1, CHUNK_OVERLAP)
results_2 = evaluate_chunk_size(chunk_size_2, CHUNK_OVERLAP)

print(f"\nСравнение размера чанка (Топ-{TOP_K} retrieval):")
print(f"\nРазмер чанка = {chunk_size_1}:")
print(f"  Число чанков: {results_1['num_chunks']}")
print(f"  Hit@{TOP_K}: {results_1['hit_rate']:.2%}")
print(f"  Recall@{TOP_K}: {results_1['recall_rate']:.2%}")

print(f"\nРазмер чанка = {chunk_size_2}:")
print(f"  Число чанков: {results_2['num_chunks']}")
print(f"  Hit@{TOP_K}: {results_2['hit_rate']:.2%}")
print(f"  Recall@{TOP_K}: {results_2['recall_rate']:.2%}")

print(f"\nВывод:")
if results_1['hit_rate'] > results_2['hit_rate']:
    print(f"  Меньшие чанки (размер={chunk_size_1}) работают лучше для hit rate")
else:
    print(f"  Большие чанки (размер={chunk_size_2}) работают лучше для hit rate")
print(f"  Компромисс: Большие чанки эффективнее, но меньшие имеют лучшую зернистость")

ЭКСПЕРИМЕНТ: ВЛИЯНИЕ РАЗМЕРА ЧАНКА

Сравнение размера чанка (Топ-5 retrieval):

Размер чанка = 256:
  Число чанков: 45
  Hit@5: 90.00%
  Recall@5: 83.33%

Размер чанка = 512:
  Число чанков: 24
  Hit@5: 80.00%
  Recall@5: 80.00%

Вывод:
  Меньшие чанки (размер=256) работают лучше для hit rate
  Компромисс: Большие чанки эффективнее, но меньшие имеют лучшую зернистость


## Секция 2.3.7: Обновление базы знаний и переиндексация

Добавить новые документы в базу знаний и показать, как меняются результаты retrieval.

In [16]:
# ============================================================================
# Обновление базы знаний
# ============================================================================

print("="*70)
print("ОБНОВЛЕНИЕ БАЗЫ ЗНАНИЙ")
print("="*70)

# Добавить новые документы в базу знаний
new_documents = [
    {
        "id": 13,
        "title": "Что такое векторная квантизация и почему она используется в FAISS?",
        "content": "Векторная квантизация уменьшает объем памяти эмбеддингов путем приближения высокомерных векторов с центроидами кластеров. В FAISS, квантизация произведений (PQ) разделяет векторное пространство на подпространства и квантует каждое подпространство независимо. Это уменьшает требования к памяти с float32 (4 байта на значение) потенциально до 1-2 байт на размерность. Компромисс - небольшая потеря в точности retrieval для гораздо более быстрого поиска и меньшего размера индекса. Квантизация произведений особенно полезна для наборов данных масштаба миллиардов. Она хорошо комбинируется с другими компонентами FAISS, такими как инвертированные файловые списки для гибридных индексов."
    },
    {
        "id": 14,
        "title": "Как вы обращаетесь с запросами вне распределения в системах retrieval?",
        "content": "Запросы вне распределения (out-of-distribution) - это те, которые не совпадают с доменом вашей базы знаний. Стандартные системы retrieval могут возвращать нерелевантные результаты с высокими баллами уверенности. Стратегии обращения с OOD запросами включают: (1) установка порога сходства - отклонение результатов ниже порога, (2) использование фильтрации запросов - выявление OOD запросов перед retrieval, (3) возврат баллов уверенности пользователям, (4) дообучение моделей эмбеддинга на специфичных для домена данных, (5) использование rerankers для фильтрации результатов. Важно мониторить вашу систему на наличие OOD запросов и понимать режимы отказа. Сбор запросов, где retrieval не удается, помогает определить пробелы в вашей базе знаний."
    },
    {
        "id": 15,
        "title": "Какие основные проблемы при масштабировании систем retrieval?",
        "content": "Масштабирование систем retrieval до миллиардов документов сталкивается с несколькими проблемами: (1) ограничения памяти - хранение эмбеддингов миллиардов документов, (2) задержка - время ответа на запрос должно остаться под миллисекунды, (3) обновления индекса - добавление/удаление документов без простоев, (4) обновление эмбеддингов - обновление эмбеддингов при изменении модели, (5) распределенный поиск - поиск по нескольким машинам. Решения включают: использование приблизительных индексов (HNSW, IVF), методы квантизации для уменьшения памяти, разделение данных по узлам, реализация добавочных обновлений и использование ускорения оборудованием (GPU). Компании, такие как Meta и Google, выпустили с открытым исходным кодом инструменты как FAISS и ScaNN для решения этих проблем в масштабе."
    }
]

# Добавить к списку документов
documents_updated = documents + new_documents
print(f"\nДобавление {len(new_documents)} новых документов в базу знаний")
print(f"  Документов до: {len(documents)}")
print(f"  Документов после: {len(documents_updated)}")

for doc in new_documents:
    print(f"  [{doc['id']}] {doc['title']}")

# Перечанкирование и переиндексация
print("\nПеречанкирование и переиндексация...")
chunks_list_updated = []
for doc in documents_updated:
    chunks = chunk_text(doc['content'], CHUNK_SIZE, CHUNK_OVERLAP)
    for chunk_idx, chunk in enumerate(chunks):
        chunks_list_updated.append({
            'doc_id': doc['id'],
            'title': doc['title'],
            'chunk_idx': chunk_idx,
            'text': chunk,
            'source': f"Doc{doc['id']}_Chunk{chunk_idx}"
        })

print(f"  Чанков до обновления: {len(chunks_list)}")
print(f"  Чанков после обновления: {len(chunks_list_updated)}")

# Генерировать эмбеддинги для обновленного корпуса
chunk_texts_updated = [chunk['text'] for chunk in chunks_list_updated]
embeddings_updated = model.encode(chunk_texts_updated, convert_to_numpy=True, show_progress_bar=False)

# Построить новый индекс FAISS
index_updated = faiss.IndexFlatL2(embeddings_updated.shape[1])
index_updated.add(embeddings_updated)

print(f"✓ Индекс перестроен с обновленной базой знаний")

def retrieve_updated(query: str, k: int = TOP_K):
    """Извлечь, используя обновленный индекс."""
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index_updated.search(query_embedding, k)
    
    results = []
    for idx, (distance, retrieved_idx) in enumerate(zip(distances[0], indices[0])):
        if retrieved_idx >= 0:
            chunk = chunks_list_updated[retrieved_idx]
            similarity = 1.0 / (1.0 + distance)
            results.append({
                'rank': idx + 1,
                'source': chunk['source'],
                'doc_id': chunk['doc_id'],
                'title': chunk['title']
            })
    return results

# ============================================================================
# Сравнить "До" и "После" обновления
# ============================================================================

print("\n" + "-"*70)
print("ПЕРЕД vs ПОСЛЕ: Влияние обновления базы знаний")
print("-"*70)

# Выбрать запросы, которые должны выиграть от новых документов
update_test_queries = [
    ("Что такое квантизация векторов?", [13]),  # Должна теперь найти Doc 13
    ("Как вы обращаетесь с запросами вне вашего домена?", [14]),  # Должна найти Doc 14
    ("Расскажи о проблемах масштабирования retrieval", [15]),  # Должна найти Doc 15
]

comparison_results = []

for query, expected_new_docs in update_test_queries:
    # До обновления
    retrieved_before = retrieve(query, k=3)
    sources_before = ', '.join([r['source'] for r in retrieved_before])
    hit_before = any(d in [r['doc_id'] for r in retrieved_before] for d in expected_new_docs)
    
    # После обновления
    retrieved_after = retrieve_updated(query, k=3)
    sources_after = ', '.join([r['source'] for r in retrieved_after])
    hit_after = any(d in [r['doc_id'] for r in retrieved_after] for d in expected_new_docs)
    
    changed = hit_before != hit_after
    
    comparison_results.append({
        'query': query,
        'before_retrieved_sources': sources_before,
        'after_retrieved_sources': sources_after,
        'hit_before': hit_before,
        'hit_after': hit_after,
        'changed': changed
    })
    
    print(f"\nЗапрос: '{query}'")
    print(f"  До:    {sources_before}")
    print(f"  После: {sources_after}")
    if changed:
        print(f"  ✓ УЛУЧШЕНО - Найден новый релевантный документ!")

# Сохранить сравнение
comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(ARTIFACTS_DIR / "retrieval_before_after_update.csv", index=False)
print(f"\n✓ Сравнение до/после сохранено в artifacts/retrieval_before_after_update.csv")

ОБНОВЛЕНИЕ БАЗЫ ЗНАНИЙ

Добавление 3 новых документов в базу знаний
  Документов до: 12
  Документов после: 15
  [13] Что такое векторная квантизация и почему она используется в FAISS?
  [14] Как вы обращаетесь с запросами вне распределения в системах retrieval?
  [15] Какие основные проблемы при масштабировании систем retrieval?

Перечанкирование и переиндексация...
  Чанков до обновления: 45
  Чанков после обновления: 57
✓ Индекс перестроен с обновленной базой знаний

----------------------------------------------------------------------
ПЕРЕД vs ПОСЛЕ: Влияние обновления базы знаний
----------------------------------------------------------------------

Запрос: 'Что такое квантизация векторов?'
  До:    Doc9_Chunk3, Doc7_Chunk3, Doc9_Chunk2
  После: Doc9_Chunk3, Doc7_Chunk3, Doc13_Chunk3
  ✓ УЛУЧШЕНО - Найден новый релевантный документ!

Запрос: 'Как вы обращаетесь с запросами вне вашего домена?'
  До:    Doc4_Chunk3, Doc7_Chunk3, Doc5_Chunk0
  После: Doc4_Chunk3, Doc14_Chunk3, Doc7

## Секция 2.3.8: Система Mini-RAG

Построить простой конвейер RAG, который извлекает контекст и генерирует ответы на основе этого контекста.

In [17]:
# ============================================================================
# Простая система Mini-RAG
# ============================================================================

print("="*70)
print("СИСТЕМА MINI-RAG")
print("="*70)

class SimpleRAG:
    """Простая система RAG с использованием retrieval и правило-based генерации."""
    
    def __init__(self, index, chunks_list_data, model_retrieval, k=TOP_K):
        self.index = index
        self.chunks_data = chunks_list_data
        self.model = model_retrieval
        self.k = k
    
    def retrieve_context(self, query: str) -> str:
        """Извлечь релевантные чанки и конкатенировать как контекст."""
        query_emb = self.model.encode([query], convert_to_numpy=True)
        distances, indices = self.index.search(query_emb, self.k)
        
        context_parts = []
        sources = []
        
        for idx in indices[0]:
            if idx >= 0:
                chunk = self.chunks_data[idx]
                context_parts.append(chunk['text'])
                sources.append(chunk['source'])
        
        context = "\n\n".join(context_parts)
        return context, sources
    
    def generate_answer(self, query: str, context: str, sources: List[str]) -> str:
        """
        Генерировать ответ на основе извлеченного контекста.
        Используем простую правило-based генерацию для этой учебной системы.
        """
        # Простая эвристическая генерация ответа
        query_lower = query.lower()
        
        if not context:
            answer = "У меня нет релевантной информации в моей базе знаний для ответа на этот вопрос."
        elif "что" in query_lower or "как" in query_lower:
            # Извлечь первое предложение из контекста как ответ
            sentences = context.split('. ')
            if sentences:
                answer = sentences[0] + "."
                # Попытаться сделать более полным
                if len(sentences) > 1:
                    answer = sentences[0] + ". " + sentences[1] + "."
            else:
                answer = context[:200] + "..."
        elif "расскажи" in query_lower or "объясни" in query_lower:
            # Использовать первые 300 символов как ответ
            answer = context[:300] + "..."
        elif "почему" in query_lower:
            answer = f"На основе базы знаний: {context[:250]}..."
        else:
            # По умолчанию: использовать первую часть контекста
            answer = context[:250] + "..."
        
        return answer
    
    def answer_question(self, question: str) -> Dict:
        """Ответить на вопрос, используя retrieval и генерацию."""
        context, sources = self.retrieve_context(question)
        answer = self.generate_answer(question, context, sources)
        
        return {
            'question': question,
            'answer': answer,
            'retrieved_sources': ', '.join(sources),
            'context': context[:500] + "..."  # Усечение для отображения
        }

# Инициализировать систему RAG с обновленной базой знаний
rag_system = SimpleRAG(index_updated, chunks_list_updated, model, k=3)

# ============================================================================
# Тестирование Mini-RAG
# ============================================================================

test_questions = [
    "Что такое word embeddings?",
    "Как работает FAISS?",
    "Расскажи о семантическом поиске",
    "Что такое RAG?",
    "Как оценивается качество retrieval?",
    "Что такое векторная квантизация?"  # новый документ
]

rag_results = []

print("\nТестирование Mini-RAG на примерах вопросов:\n")

for i, question in enumerate(test_questions, 1):
    result = rag_system.answer_question(question)
    rag_results.append({
        'question': result['question'],
        'answer': result['answer'],
        'retrieved_sources': result['retrieved_sources']
    })
    
    print(f"{i}. Вопрос: {question}")
    print(f"   Источники: {result['retrieved_sources']}")
    print(f"   Ответ: {result['answer'][:150]}...")
    print()

# Сохранить примеры RAG
rag_df = pd.DataFrame(rag_results)
rag_df.to_csv(ARTIFACTS_DIR / "rag_examples.csv", index=False)
print(f"✓ Примеры RAG сохранены в artifacts/rag_examples.csv")

СИСТЕМА MINI-RAG

Тестирование Mini-RAG на примерах вопросов:

1. Вопрос: Что такое word embeddings?
   Источники: Doc2_Chunk2, Doc2_Chunk0, Doc1_Chunk0
   Ответ: теризации документов и семантического поиска. Современные sentence embeddings из моделей, таких как BERT и Sentence-BERT, производят лучшие представле...

2. Вопрос: Как работает FAISS?
   Источники: Doc4_Chunk3, Doc13_Chunk3, Doc3_Chunk0
   Ответ: мбеддингом запроса. Это дает нам наиболее семантически похожие документы.

кими как инвертированные файловые списки для гибридных индексов.

FAISS (Fa...

3. Вопрос: Расскажи о семантическом поиске
   Источники: Doc4_Chunk3, Doc9_Chunk3, Doc13_Chunk3
   Ответ: мбеддингом запроса. Это дает нам наиболее семантически похожие документы.

ия контекста), семантическое чанкирование на основе предложений или абзацев...

4. Вопрос: Что такое RAG?
   Источники: Doc4_Chunk3, Doc7_Chunk2, Doc9_Chunk3
   Ответ: мбеддингом запроса. Это дает нам наиболее семантически похожие документы.

тексте...

## Секция 2.3.9: Анализ ошибок

Проанализировать, где система mini-RAG делает ошибки, и обсудить потенциальные улучшения.

In [18]:
# ============================================================================
# Анализ ошибок: Неудачные случаи и улучшения
# ============================================================================

print("="*70)
print("АНАЛИЗ ОШИБОК - Определение ограничений системы")
print("="*70)

# Тестовые случаи, разработанные для определения режимов отказа
test_cases = [
    {
        "question": "Как я могу конфигурировать квантовое туннелирование в нейронных сетях?",
        "type": "вне_домена",
        "explanation": "Этот запрос полностью выходит за пределы области базы знаний"
    },
    {
        "question": "xyz abc qwerty",
        "type": "бессмысленный",
        "explanation": "Полностью бессмысленный ввод"
    },
    {
        "question": "получить информацию",
        "type": "расплывчатый_и_общий",
        "explanation": "Очень расплывчатый запрос, который может совпадать со многими документами плохо"
    },
    {
        "question": "Какая связь между эмбеддингами и моделями трансформер?",
        "type": "частичное_покрытие",
        "explanation": "База знаний содержит эмбеддинги, но ограниченное покрытие трансформеров"
    }
]

print("\nТестирование граничных случаев и режимов отказа:\n")

for i, test_case in enumerate(test_cases, 1):
    question = test_case['question']
    context, sources = rag_system.retrieve_context(question)
    answer = rag_system.generate_answer(question, context, sources)
    
    print(f"{i}. Тип: {test_case['type'].upper()}")
    print(f"   Вопрос: {question}")
    print(f"   Извлеченные источники: {sources}")
    print(f"   Проблема: {test_case['explanation']}")
    print()

print("="*70)
print("СВОДКА ОГРАНИЧЕНИЙ И УЛУЧШЕНИЙ")
print("="*70)

limitations = [
    {
        "limitation": "Ограниченное покрытие базы знаний",
        "example": "Вопросы о GPU, TPU или распределенном обучении возвращают нерелевантные результаты",
        "solution": "Расширить базу знаний с проблемами на эти темы"
    },
    {
        "limitation": "Генерация ответов основана на правилах и упрощена",
        "example": "Ответы - это просто усеченный контекст без фактического синтеза",
        "solution": "Использовать языковую модель (как BERT или GPT) для генерации свободного ответа"
    },
    {
        "limitation": "Отсутствует обработка многочастевых вопросов",
        "example": "Сложные вопросы с несколькими подвопросами не разложены",
        "solution": "Реализовать разложение запроса и динамический retrieval"
    },
    {
        "limitation": "Нет переупорядочивания извлеченных документов",
        "example": "Первые несколько результатов retrieval просто конкатенируются",
        "solution": "Добавить модуль переупорядочивания для переупорядочения результатов по релевантности"
    },
    {
        "limitation": "Нет цикла обратной связи для запросов вне домена",
        "example": "Система не различает уверенные и неуверенные ответы",
        "solution": "Отслеживать и анализировать неудачные запросы, чтобы обновить базу знаний"
    }
]

for i, item in enumerate(limitations, 1):
    print(f"\n{i}. ОГРАНИЧЕНИЕ: {item['limitation']}")
    print(f"   Пример: {item['example']}")
    print(f"   Улучшение: {item['solution']}")

print("\n" + "="*70)
print("СВОДКА ОЦЕНКИ RETRIEVAL")
print("="*70)
print(f"\nРезультаты оценки на контрольных запросах:")
print(f"  Всего запросов: {len(evaluation_results)}")
print(f"  Hit@{TOP_K}: {hit_rate:.1%}")
print(f"  Recall@{TOP_K}: {recall_rate:.1%}")
print(f"  MRR@{TOP_K}: {mrr_rate:.3f}")
print(f"\nСистема достигает хорошей базовой производительности на запросах в домене,")
print(f"но улучшения в генерации ответов и контекстуализации помогут.")
print(f"\nКлючевой вывод: Векторный retrieval + чанкирование работает хорошо,")
print(f"но качество генерации зависит от результатов retrieval.")

АНАЛИЗ ОШИБОК - Определение ограничений системы

Тестирование граничных случаев и режимов отказа:

1. Тип: ВНЕ_ДОМЕНА
   Вопрос: Как я могу конфигурировать квантовое туннелирование в нейронных сетях?
   Извлеченные источники: ['Doc13_Chunk3', 'Doc7_Chunk3', 'Doc5_Chunk2']
   Проблема: Этот запрос полностью выходит за пределы области базы знаний

2. Тип: БЕССМЫСЛЕННЫЙ
   Вопрос: xyz abc qwerty
   Извлеченные источники: ['Doc1_Chunk2', 'Doc3_Chunk3', 'Doc8_Chunk3']
   Проблема: Полностью бессмысленный ввод

3. Тип: РАСПЛЫВЧАТЫЙ_И_ОБЩИЙ
   Вопрос: получить информацию
   Извлеченные источники: ['Doc4_Chunk3', 'Doc7_Chunk3', 'Doc13_Chunk3']
   Проблема: Очень расплывчатый запрос, который может совпадать со многими документами плохо

4. Тип: ЧАСТИЧНОЕ_ПОКРЫТИЕ
   Вопрос: Какая связь между эмбеддингами и моделями трансформер?
   Извлеченные источники: ['Doc4_Chunk3', 'Doc13_Chunk3', 'Doc7_Chunk3']
   Проблема: База знаний содержит эмбеддинги, но ограниченное покрытие трансформеров

СВОДКА ОГР